# Phase 3 — Full Training v2 (QLoRA Qwen3.5-4B-Base)

**Muc dich:** Full train 85K samples, validate improvement vs v1 smoke.  
**Design:** Section 5 `plan_day5.md`. Framework: PEFT + bitsandbytes 4-bit NF4.  
**Fix tu v1:** `torch_dtype=torch.bfloat16` trong `from_pretrained` (tranh conv1d bf16/float32 crash).  

| Param | v1 smoke | v2 full |
|-------|----------|---------|
| Train size | 20,000 | **85,727 (full)** |
| LoRA rank | 32 | **64** |
| LoRA alpha | 64 | **128** |
| Target modules | 4 (attention) | **7 (attn + MLP)** |
| Epochs | 2 | **3** |
| Val eval | 500 | **200** |

**Ky vong RMSLE:** < 0.50 (beat v1=0.6084). Muc tieu: < 0.4004 (beat Day4 v8).

In [ ]:
# Chay neu chua co trong env:
#!uv add "transformers>=5.2.0" peft trl bitsandbytes accelerate datasets python-dotenv

In [ ]:
import os
import re
import sys
import gc
import json
import time
import glob
import math
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
import bitsandbytes as bnb
import torch.nn as nn
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

from dataclasses import dataclass
from typing import Any, Dict, List
from transformers import PreTrainedTokenizerBase

@dataclass
class DataCollatorForCompletionOnlyLM:
    """Manual impl: trl.DataCollatorForCompletionOnlyLM removed in TRL 0.24.0."""
    response_template: List[int]
    tokenizer: PreTrainedTokenizerBase
    ignore_index: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in input_ids_list)
        bs = len(input_ids_list)

        padded = torch.full((bs, max_len), self.tokenizer.pad_token_id, dtype=torch.long)
        attn   = torch.zeros((bs, max_len), dtype=torch.long)
        labels = torch.full((bs, max_len), self.ignore_index, dtype=torch.long)

        tpl, tpl_len = self.response_template, len(self.response_template)

        for i, ids in enumerate(input_ids_list):
            n = len(ids)
            padded[i, :n] = ids
            attn[i, :n]   = 1
            for j in range(n - tpl_len, -1, -1):
                if ids[j : j + tpl_len].tolist() == tpl:
                    labels[i, j + tpl_len : n] = ids[j + tpl_len : n]
                    break

        return {"input_ids": padded, "attention_mask": attn, "labels": labels}

print("DataCollatorForCompletionOnlyLM: manual impl OK")

NOTEBOOK_DIR = Path("__file__").parent if "__file__" in dir() else Path(".")
sys.path.insert(0, str(NOTEBOOK_DIR))
from utils.evaluator import compute_metrics, plot_predictions

print("Imports OK")
import transformers, peft, trl
for pkg, mod in [("torch", torch), ("transformers", transformers), ("peft", peft), ("trl", trl)]:
    print(f"  {pkg:<14}: {mod.__version__}")

In [ ]:
# --- Section 5 constants (v2 full) ---

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME = "SeanSunny/items_prompts_tv_3"

# Sequence (khong thay doi tu v1)
MAX_SEQ_LENGTH  = 192
MAX_NEW_TOKENS  = 4
QUESTION_PREFIX = "S\u1ea3n ph\u1ea9m n\u00e0y c\u00f3 gi\u00e1 bao nhi\u00eau ?\n"
PRICE_PREFIX    = "\n\nGi\u00e1 l\u00e0: "

# LoRA (v2: r=64, alpha=128, 7 modules = attn + MLP)
LORA_R              = 64
LORA_ALPHA          = 128
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]

# Training (v2 full)
TRAIN_SIZE        = None          # None = full 85,727
VAL_EVAL_SIZE     = 200
NUM_EPOCHS        = 3
PER_DEVICE_BATCH  = 16            # same as v1 actual run
GRAD_ACCUM        = 4             # eff batch = 64
LEARNING_RATE     = 2e-4
LR_SCHEDULER      = "cosine"
WARMUP_RATIO      = 0.03
WEIGHT_DECAY      = 0.001
OPTIM             = "paged_adamw_32bit"
PACKING           = False
GRADIENT_CHECKPOINTING = True     # bat lai v2 (r=64 + 7 modules tốn VRAM hon)
EVAL_STEPS        = 200           # it hon (full 85K moi step la 85K/64=1340 steps/ep)
SAVE_STRATEGY     = "epoch"
LOGGING_STEPS     = 50
SEED              = 42

# Inference safety (khong thay doi)
PRED_CLAMP_MIN = 5
PRED_CLAMP_MAX = 1000
PARSE_REGEX    = r"[-+]?\d*\.\d+|\d+"

# Paths
ADAPTER_DIR     = NOTEBOOK_DIR / "weights" / "v2_adapter"
RESULTS_FILE    = NOTEBOOK_DIR / "results" / "v2_results.json"
HF_REPO_ADAPTER = "SeanSunny/qwen3.5-4b-vn-pricer-v2"

print(f"BASE_MODEL    : {BASE_MODEL}")
print(f"DATASET_NAME  : {DATASET_NAME}")
print(f"LoRA          : r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES}")
print(f"Training      : full data, {NUM_EPOCHS} epochs, eff_batch={PER_DEVICE_BATCH*GRAD_ACCUM}")
print(f"ADAPTER_DIR   : {ADAPTER_DIR}")
print(f"RESULTS_FILE  : {RESULTS_FILE}")

In [ ]:
# GPU check + HF login + dirs
assert torch.cuda.is_available(), "GPU khong kha dung."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
cap = torch.cuda.get_device_capability()
print(f"bf16 : {'yes' if cap[0] >= 8 else 'no'} (compute {cap})")

env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK (from {env_path})")
else:
    print(f"HF_TOKEN not set in {env_path}")
    login()

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
(NOTEBOOK_DIR / "results").mkdir(exist_ok=True)
print("Dirs ready: weights/v2_adapter/, results/")

## 1. Load model + tokenizer

**Fix v1 bug:** `torch_dtype=torch.bfloat16` dam bao tat ca non-quantized layers (conv1d, layernorm...)  
duoc load o bf16 thay vi float32. Tranh `RuntimeError: expected BFloat16 but found Float`  
tai `Qwen3_5GatedDeltaNet.conv1d` trong inference.  
**R4:** `prepare_model_for_kbit_training` TRUOC `get_peft_model`.  
**R5:** Verify EOS/PAD tokens.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# R5: verify EOS/PAD
if tokenizer.eos_token_id is None:
    tokenizer.eos_token = "<|endoftext|>"
    print("WARNING: set eos_token manually to <|endoftext|>")
print(f"EOS token : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"PAD token : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # Fix: dam bao conv1d va non-quant layers o bf16
)

# R4: prepare TRUOC khi apply LoRA
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING)
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 2. Verify Qwen3.5 modules + apply LoRA

**v2:** 7 modules = 4 attention (`q/k/v/o_proj`) + 3 MLP (`gate/up/down_proj`).  
MLP modules giup model hoc bien doi features, khong chi attention pattern.  
**R2:** Verify runtime truoc khi apply, fallback `all-linear` neu thieu.

In [ ]:
linear_suffixes = set()
for name, module in model.named_modules():
    if isinstance(module, (nn.Linear, bnb.nn.Linear4bit)):
        linear_suffixes.add(name.split(".")[-1])

print("Linear module suffixes found:", sorted(linear_suffixes))

EXPECTED_V2 = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
if EXPECTED_V2.issubset(linear_suffixes):
    target_modules = LORA_TARGET_MODULES
    print(f"PASS: all 7 modules found. target_modules = {target_modules}")
else:
    target_modules = "all-linear"
    missing = EXPECTED_V2 - linear_suffixes
    print(f"WARNING: missing {missing}. Fallback target_modules = 'all-linear'")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.enable_input_require_grads()

## 3. Dataset + truncation analysis

**R7:** Do p50/p95/p99 summary token len, log truncation rate.  
Train = full 85,727. Val eval = 200 sample (smoke nhanh).

In [ ]:
ds = load_dataset(DATASET_NAME)
print(f"Train: {len(ds['train']):,} | Val: {len(ds['val']):,} | Test: {len(ds['test']):,}")

# Compute TOKENS_FIXED runtime
q_ids = tokenizer.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
TOKENS_FIXED = len(q_ids) + len(p_ids)
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2

print(f"TOKENS_FIXED        : {TOKENS_FIXED}")
print(f"MAX_SUMMARY_TOKENS  : {MAX_SUMMARY_TOKENS}")

# Lay full train + 200 val raw
train_full_raw = ds["train"].shuffle(seed=SEED)
val_raw = ds["val"].shuffle(seed=SEED).select(range(VAL_EVAL_SIZE))

# R7: do distribution summary token len tren 5000 train sample (uoc luong nhanh)
sample_for_profile = train_full_raw.select(range(min(5000, len(train_full_raw))))
summaries_profile = []
for item in sample_for_profile:
    p = item["prompt"]
    summaries_profile.append(p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)])

summary_lens = np.array(
    [len(tokenizer.encode(s, add_special_tokens=False)) for s in tqdm(summaries_profile, desc="Profiling 5K summaries")]
)
n_truncated_est = int((summary_lens > MAX_SUMMARY_TOKENS).sum())

print(f"\nSummary token len (5K sample):")
print(f"  p50={np.percentile(summary_lens, 50):.0f}, "
      f"p95={np.percentile(summary_lens, 95):.0f}, "
      f"p99={np.percentile(summary_lens, 99):.0f}, "
      f"max={summary_lens.max()}")
print(f"Truncated (est): {n_truncated_est}/{len(summaries_profile)} ({n_truncated_est/len(summaries_profile)*100:.1f}%)")
if n_truncated_est / len(summaries_profile) > 0.5:
    print("WARNING: truncation rate > 50%. Xem lai MAX_SUMMARY_TOKENS.")

In [ ]:
# Preprocess: cat summary token-level tu duoi (Q2/R7)
def preprocess(example):
    p = example["prompt"]
    summary = p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)]
    summary_ids = tokenizer.encode(summary, add_special_tokens=False)
    if len(summary_ids) > MAX_SUMMARY_TOKENS:
        summary_ids = summary_ids[:MAX_SUMMARY_TOKENS]
        summary = tokenizer.decode(summary_ids, skip_special_tokens=True).rstrip()
    full_text = QUESTION_PREFIX + summary + PRICE_PREFIX + example["completion"] + "\n" + tokenizer.eos_token
    return {"text": full_text}

train_ds = train_full_raw.map(preprocess, desc="Preprocess train")
val_ds   = val_raw.map(preprocess, desc="Preprocess val")

print(f"train_ds: {len(train_ds):,} | val_ds: {len(val_ds):,}")

# Eyeball 2 samples
for i in range(2):
    t = train_ds[i]["text"]
    print(f"--- Sample {i} ---")
    print(repr(t[:200]) + ("..." if len(t) > 200 else ""))
    assert PRICE_PREFIX in t, f"ERROR: PRICE_PREFIX missing in sample {i}"
print("PRICE_PREFIX present: OK")

In [ ]:
# Pre-tokenize (TRL 0.24.0 khong con dataset_text_field)
def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

train_ds_tok = train_ds.map(tokenize_fn, batched=False, remove_columns=["text"])
val_ds_tok   = val_ds.map(tokenize_fn, batched=False, remove_columns=["text"])

print(f"Columns: {train_ds_tok.column_names}")
print(f"Sample input_ids len: {len(train_ds_tok[0]['input_ids'])}")

## 4. DataCollator + verify mask

**R1:** Token IDs, khong phai string.  
**R6:** Fail-loud neu mask sai — dung notebook, khong train.

In [ ]:
# R1: encode PRICE_PREFIX thanh token IDs
response_template_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
decoded_back = tokenizer.decode(response_template_ids)
print(f"response_template_ids : {response_template_ids}")
print(f"decoded back          : {decoded_back!r}")
print(f"matches PRICE_PREFIX  : {decoded_back == PRICE_PREFIX}")

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

# R6: verify mask fail-loud
sample_text = train_ds[0]["text"]
tokenized = tokenizer(
    sample_text, return_tensors="pt",
    max_length=MAX_SEQ_LENGTH, truncation=True,
)
batch_in = [{
    "input_ids": tokenized["input_ids"][0].tolist(),
    "attention_mask": tokenized["attention_mask"][0].tolist(),
}]
batch_out = collator(batch_in)
labels = batch_out["labels"][0]
non_masked = labels[labels != -100]
decoded_labels = tokenizer.decode(non_masked.tolist(), skip_special_tokens=False)

print(f"\nNon-masked token count : {len(non_masked)}")
print(f"Decoded non-masked     : {decoded_labels!r}")
print(f"Sample completion      : {train_full_raw[0]['completion']!r}")

if len(non_masked) == 0:
    raise RuntimeError(
        "Mask verify FAILED: response_template_ids not found. Abort training."
    )

expected_max_tokens = MAX_NEW_TOKENS + 2
if len(non_masked) > expected_max_tokens + 2:
    print(f"WARNING: non-masked count {len(non_masked)} > expected {expected_max_tokens}. "
          "Co the co prompt leak.")
else:
    print("\nMask verify PASS: chi thay completion + EOS trong labels.")

## 5. VRAM smoke (100 samples)

**R8:** Train 100 samples, max_steps=5 — verify VRAM voi config r=64 + 7 modules + gradient_checkpointing=True.  
Threshold: < 23 GB. Fallback neu OOM: `PER_DEVICE_BATCH=8, GRAD_ACCUM=8`.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t_smoke_start = time.time()

trainer_smoke = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_tok.select(range(100)),
    data_collator=collator,
    args=SFTConfig(
        output_dir=str(NOTEBOOK_DIR / "weights" / "v2_smoke_run"),
        max_steps=5,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        bf16=True,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
    ),
)
trainer_smoke.train()

vram_smoke_gb = torch.cuda.max_memory_allocated() / 1e9
t_smoke = time.time() - t_smoke_start
sec_per_step = t_smoke / 5

steps_per_epoch = math.ceil(len(train_ds_tok) / (PER_DEVICE_BATCH * GRAD_ACCUM))
total_steps = steps_per_epoch * NUM_EPOCHS
est_total_sec = sec_per_step * total_steps

print(f"\nVRAM peak   : {vram_smoke_gb:.2f} GB")
print(f"Sec/step    : {sec_per_step:.2f}s")
print(f"Steps/epoch : {steps_per_epoch}")
print(f"Total steps : {total_steps}")
print(f"Est. total  : {est_total_sec/60:.1f} min ({est_total_sec/3600:.1f} hr)")

if vram_smoke_gb > 23.0:
    print(f"\nWARNING: VRAM {vram_smoke_gb:.1f} GB > 23 GB. Fallback: PER_DEVICE_BATCH=8, GRAD_ACCUM=8.")
    print("  -> Sua lai 2 hang tren roi chay lai tu cell nay.")
else:
    print(f"\nVRAM OK ({vram_smoke_gb:.1f} GB < 23 GB).")

print("\n[CHECKPOINT] Smoke OK. Proceed to full train cell.")

In [ ]:
# Cleanup sau smoke — giai phong VRAM truoc khi train full
del trainer_smoke
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print(f"VRAM reserved     : {torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

## 6. Full train 85K — 3 epochs

**Q4 B+:** `eval_strategy="steps"` CE loss in-train.  
**Q8:** `report_to="none"` — console only.  
`save_strategy="epoch"` — 3 checkpoints, eval per-epoch sau train.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t_train_start = time.time()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    data_collator=collator,
    args=SFTConfig(
        output_dir=str(ADAPTER_DIR),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        optim=OPTIM,
        bf16=True,
        max_grad_norm=0.3,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy=SAVE_STRATEGY,
        save_total_limit=4,
        logging_steps=LOGGING_STEPS,
        report_to="none",
        seed=SEED,
    ),
)
trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

total_train_sec = time.time() - t_train_start
final_vram_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"Training complete: {total_train_sec/60:.1f} min ({total_train_sec/3600:.1f} hr) | VRAM peak: {final_vram_peak:.2f} GB")

log_history = trainer.state.log_history
train_losses = [(int(e["step"]), float(e["loss"])) for e in log_history if "loss" in e and "eval_loss" not in e]
eval_losses  = [(int(e["step"]), float(e["eval_loss"])) for e in log_history if "eval_loss" in e]

print(f"Train loss entries: {len(train_losses)} | Eval loss entries: {len(eval_losses)}")
if train_losses:
    print(f"Train loss: first={train_losses[0][1]:.4f} -> last={train_losses[-1][1]:.4f}")
if eval_losses:
    print(f"Eval CE loss: first={eval_losses[0][1]:.4f} -> last={eval_losses[-1][1]:.4f}")

In [ ]:
# Cleanup sau training — giai phong cache truoc eval
# Giu model trong memory (can cho eval)
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after train cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## 7. Generative eval — 200 val (final epoch 3)

**Q5 lai:** Regex float-first + clamp [5, 1000].  
**Q6:** 200 val — nhanh (~3 phut), du de validate improvement vs v1.

In [ ]:
model.eval()

def predict_one(prompt: str) -> tuple:
    """Returns (pred_thousands_vnd, raw_generated_text)."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(PARSE_REGEX, gen)
    if m:
        pred_k = int(float(m.group()))
        pred_k = max(PRED_CLAMP_MIN, min(pred_k, PRED_CLAMP_MAX))
    else:
        pred_k = 0
    return pred_k, gen

preds_vnd, trues_vnd, raw_outs = [], [], []
clamp_count = 0
t_eval_start = time.time()

for item in tqdm(val_raw, desc="Generative eval val"):
    pred_k, raw = predict_one(item["prompt"])
    preds_vnd.append(pred_k * 1000)
    trues_vnd.append(item["price_vnd_true"])
    raw_outs.append(raw)
    if pred_k in (PRED_CLAMP_MIN, PRED_CLAMP_MAX):
        clamp_count += 1

t_eval = time.time() - t_eval_start
metrics_final = compute_metrics(np.array(trues_vnd, dtype=float), np.array(preds_vnd, dtype=float))

print("=" * 50)
print(f"v2 Full — {VAL_EVAL_SIZE} val (epoch {NUM_EPOCHS} final)")
print("=" * 50)
print(f"RMSLE : {metrics_final['rmsle']:.4f}  (primary)")
print(f"MAE   : {metrics_final['mae']:,.0f} VND")
print(f"MAPE  : {metrics_final['mape']:.1f}%")
print(f"R2    : {metrics_final['r2']:.4f}")
print(f"Zero preds   : {preds_vnd.count(0)}")
print(f"Clamp trigger: {clamp_count}")
print(f"Sec/item     : {t_eval/VAL_EVAL_SIZE:.2f}s")
print("=" * 50)
print(f"v1 reference : RMSLE=0.6084")
print(f"Day4 v8 ref  : RMSLE=0.4004")
print(f"Target       : RMSLE<0.38")

names = [item["prompt"][:50] for item in val_raw]
plot_predictions(
    np.array(trues_vnd, dtype=float),
    np.array(preds_vnd, dtype=float),
    title=f"v2 Full ({VAL_EVAL_SIZE} val)",
    names=names,
)

In [ ]:
# Cleanup sau generative eval
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after eval cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## 8. Manual checkpoint eval per-epoch (3 checkpoints)

**Q4 bonus:** Glob auto-detect, sort by step. Load tung adapter, eval 200 val, so sanh RMSLE e1/e2/e3.  
Giup quyet dinh epoch nao tot nhat cho v3.

In [ ]:
checkpoint_dirs = sorted(
    glob.glob(str(ADAPTER_DIR / "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]),
)
print(f"Checkpoints found: {len(checkpoint_dirs)}")
for d in checkpoint_dirs:
    print(f"  {d}")

metrics_per_epoch = {}

for ep_idx, ckpt_dir in enumerate(checkpoint_dirs):
    ep_num = ep_idx + 1
    print(f"\nEval checkpoint {Path(ckpt_dir).name} (epoch {ep_num})...")

    ckpt_model = PeftModel.from_pretrained(model.base_model.model, ckpt_dir)
    ckpt_model.eval()

    preds_ckpt, trues_ckpt = [], []
    for item in tqdm(val_raw, desc=f"Epoch {ep_num}", leave=False):
        inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
        with torch.inference_mode():
            out = ckpt_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        match = re.search(PARSE_REGEX, gen)
        if match:
            pk = max(PRED_CLAMP_MIN, min(int(float(match.group())), PRED_CLAMP_MAX))
        else:
            pk = 0
        preds_ckpt.append(pk * 1000)
        trues_ckpt.append(item["price_vnd_true"])

    m_ep = compute_metrics(np.array(trues_ckpt, dtype=float), np.array(preds_ckpt, dtype=float))
    metrics_per_epoch[f"epoch_{ep_num}"] = m_ep
    print(f"  Epoch {ep_num}: RMSLE={m_ep['rmsle']:.4f}, MAE={m_ep['mae']:,.0f}, MAPE={m_ep['mape']:.1f}%")

    # Cleanup sau moi epoch
    del ckpt_model
    gc.collect()
    torch.cuda.empty_cache()

print("\nEpoch comparison:")
for ep_key, m in metrics_per_epoch.items():
    print(f"  {ep_key}: RMSLE={m['rmsle']:.4f}, MAE={m['mae']:,.0f}, MAPE={m['mape']:.1f}%")

best_ep = min(metrics_per_epoch, key=lambda k: metrics_per_epoch[k]["rmsle"])
print(f"\nBest epoch: {best_ep} (RMSLE={metrics_per_epoch[best_ep]['rmsle']:.4f})")

## 9. Save + push

**Q7:** `push_to_hub` private. Schema v2_results.json day du.

In [ ]:
samples_out = []
for i in range(min(20, len(val_raw))):
    tv = trues_vnd[i]
    pv = preds_vnd[i]
    err_pct = abs(pv - tv) / tv * 100 if tv > 0 else None
    samples_out.append({
        "idx": i,
        "prompt_excerpt": val_raw[i]["prompt"][:120],
        "generated_raw": raw_outs[i],
        "pred_vnd": pv,
        "true_vnd": tv,
        "error_pct": round(err_pct, 1) if err_pct is not None else None,
    })

results = {
    "version": "v2_full",
    "model": BASE_MODEL,
    "dataset": DATASET_NAME,
    "config": {
        "train_size": len(train_ds_tok),
        "val_eval_size": VAL_EVAL_SIZE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "num_epochs": NUM_EPOCHS,
        "per_device_batch": PER_DEVICE_BATCH,
        "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "max_seq_length": MAX_SEQ_LENGTH,
        "tokens_fixed": TOKENS_FIXED,
        "max_summary_tokens": MAX_SUMMARY_TOKENS,
    },
    "vram_smoke_gb": round(vram_smoke_gb, 2),
    "vram_train_peak_gb": round(final_vram_peak, 2),
    "total_train_sec": round(total_train_sec, 1),
    "sec_per_val_item": round(t_eval / VAL_EVAL_SIZE, 2),
    "train_loss_curve": train_losses,
    "eval_loss_curve": eval_losses,
    "metrics_final": metrics_final,
    "metrics_per_epoch": metrics_per_epoch,
    "best_epoch": best_ep,
    "samples_20": samples_out,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved: {RESULTS_FILE}")

In [ ]:
# Push adapter + tokenizer len HF
print(f"Pushing adapter to {HF_REPO_ADAPTER} (private)...")
model.push_to_hub(HF_REPO_ADAPTER, private=True)
tokenizer.push_to_hub(HF_REPO_ADAPTER, private=True)
print(f"Pushed: https://huggingface.co/{HF_REPO_ADAPTER}")

In [ ]:
# Final cleanup
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM final: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## Leaderboard Day 5

In [ ]:
v0_rmsle = 4.4428
v1_rmsle = 0.6084
v2_rmsle = metrics_final["rmsle"]

print(f"{'Version':<30} {'RMSLE':>8} {'MAE':>14} {'MAPE':>8} {'R2':>8}")
print("-" * 72)
print(f"{'v0 zero-shot':<30} {v0_rmsle:>8.4f} {'296,807':>14} {'105.9%':>8} {'-2.0932':>8}")
print(f"{'v1 smoke 20K/2ep':<30} {v1_rmsle:>8.4f} {'116,769':>14} {'50.4%':>8} {'0.3980':>8}")
for ep_key, m in metrics_per_epoch.items():
    label = f"v2 full ({ep_key})"
    print(f"{label:<30} {m['rmsle']:>8.4f} {m['mae']:>14,.0f} {m['mape']:>7.1f}% {m['r2']:>8.4f}")
print(f"{'v2 full (final ep3)':<30} {v2_rmsle:>8.4f} "
      f"{metrics_final['mae']:>14,.0f} {metrics_final['mape']:>7.1f}% {metrics_final['r2']:>8.4f}")
print(f"{'v8 Day4 (ref)':<30} {'0.4004':>8} {'79,853':>14} {'30.7%':>8} {'0.6920':>8}")
print()
print(f"Improvement v1 -> v2 final: {v1_rmsle - v2_rmsle:+.4f}")
print(f"Gap vs Day4 v8            : {v2_rmsle - 0.4004:+.4f}")
print(f"Gap vs target 0.38        : {v2_rmsle - 0.38:+.4f}")
print()
print("Buoc tiep: Phase 4 — v3 high-rank (05_train_v3.ipynb)" if v2_rmsle > 0.38 else "Target 0.38 ACHIEVED! Buoc tiep: Phase 5 v4 tricks.")